# 02 — Group Study (multivariate Optuna ablation)

11축 동시 탐색 (TPE multivariate). LGBM HP는 default 고정 → 축 효과 + 상호작용만 학습.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*.csv`
- **출력**: `4_output/0_baseline/group/{optuna.db, trials.csv, param_importance.csv}`
- **참조**: [strategy.md §5](strategy.md), [strategy_common.md §4·§6·§8](../strategy_common.md)

## 1. 환경 설정 + 데이터 로드

In [1]:
import os, sys

# Colab이면 코드/데이터 zip을 Drive에서 받아 풀고 PROJECT_ROOT를 잡음, 로컬이면 ../../setup.py만
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
    PROJECT_ROOT = '/content/project'
except ImportError:
    %run ../../setup.py
    from utils.config import PROJECT_ROOT

# 0_baseline 폴더를 경로에 추가 → `import axes` 가 이 노트북 옆의 axes.py를 찾게
BASELINE_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '0_baseline')
if BASELINE_DIR not in sys.path:
    sys.path.insert(0, BASELINE_DIR)

import warnings
warnings.filterwarnings('ignore')

from utils.data import load_all, get_feat_cols, split_xs
import axes

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
axes.set_data(xs, xs_dict, ys, feat_cols)   # run_one들이 공유할 데이터를 모듈에 1회 주입

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

setup 완료
[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
Feature 수: 1087
Die 수: train=104,748, val=34,908, test=34,916


## 2. Optuna study 설정

- Sampler: TPE (seed=None, multivariate=True, group=True) — strategy_common §4
- Pruner: 비활성 (5-fold 다 끝나야 점수 나옴)
- Storage: sqlite (4_output/0_baseline/group/optuna.db)
- Trial: default 300 (strategy.md §5.4)

In [2]:
# 노트북 상단 단일 파라미터
N_JOBS = 5         # 모델 학습 병렬도
N_TRIALS = 300     # Optuna trial 수
TIMEOUT_SEC = None  # 초 단위, None=무제한 (Colab 타임아웃 대비 시 숫자로)
N_ESTIMATORS = 100 # LGBM n_estimators 고정 — HP가 아니라 전처리 축 효과만 본다

# group study용 축 = OAT 11축 그대로, 단 impute에서 'knn' 제외.
#   knn-impute는 fit당 ~70분(spatial/median의 20~25배) → 300 trial이 며칠 걸리고,
#   impute 축 marginal 효과는 ≈1e-6로 트리 모델엔 거의 무관. knn은 OAT에서 1셀로만 측정한다.
GROUP_AXES = {**axes.AXES, 'impute': ['spatial', 'median']}

import json
import optuna
from datetime import datetime
from optuna.samplers import TPESampler

OUT_DIR = os.path.join(PROJECT_ROOT, '4_output', '0_baseline', 'group')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, 'optuna.db')
META_JSON = os.path.join(OUT_DIR, 'meta.json')
STORAGE_URL = f'sqlite:///{DB_PATH}'

# TPE: multivariate=축 간 결합 분포 학습, group=CLF=off 같은 조건부 축을 자동으로 건너뜀. seed=None → run마다 다양성
sampler = TPESampler(
    seed=None,
    multivariate=True,
    group=True,
)

study = optuna.create_study(
    study_name='baseline_group',
    storage=STORAGE_URL,
    sampler=sampler,
    direction='minimize',
    load_if_exists=True,   # 같은 db가 있으면 이어서 (끊김 후 재실행 대비)
)

# meta.json — 이번 run의 study 설정 + reference/축/고정 전처리 등을 박제
meta = {
    'created':       datetime.now().isoformat(timespec='seconds'),
    'n_jobs':        N_JOBS,
    'n_trials':      N_TRIALS,
    'n_estimators':  N_ESTIMATORS,
    'study_name':    'baseline_group',
    'sampler':       'TPESampler(seed=None, multivariate=True, group=True)',
    'pruner':        'None (5-fold complete eval)',
    'direction':     'minimize',
    'objective':     'oof_rmse',
    'reference':     axes.REFERENCE,
    'axes':          {k: list(map(str, v)) for k, v in GROUP_AXES.items()},
    'axes_note':     "impute에서 'knn' 제외 (fit당 ~70분 — OAT에서만 측정). 그 외는 axes.AXES와 동일",
    'agg_preset_lib': axes.AGG_PRESET_LIB,
    'pp_pin': {
        'cleaning': axes.PP_PIN_CLEANING,
        'outlier':  axes.PP_PIN_OUTLIER,
        'binarize': axes.PP_PIN_BINARIZE,
        'iso':      axes.PP_PIN_ISO,
        'lds':      axes.PP_PIN_LDS,
        'ge':       axes.PP_PIN_GE,
    },
    'exclude_cols':  axes.EXCLUDE_COLS,
}
with open(META_JSON, 'w') as f:
    json.dump(meta, f, indent=2, default=str, ensure_ascii=False)

print(f'Study : baseline_group')
print(f'Storage: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')
print(f'meta.json saved → {META_JSON}')


[I 2026-05-11 17:19:49,822] A new study created in RDB with name: baseline_group


Study : baseline_group
Storage: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\baseline\group\optuna.db
기존 trial: 0
meta.json saved → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\baseline\group\meta.json


## 3. Objective + study.optimize

GROUP_AXES(=axes.AXES, 단 impute는 knn 제외)의 categorical을 trial.suggest_categorical로 샘플 → axes.run_one(cfg, seed=42) 호출. objective = OOF RMSE (train 5-fold CV).

In [3]:
def objective(trial):
    # 전처리 축을 각각 categorical로 샘플 (GROUP_AXES = OAT 11축, 단 impute는 knn 제외)
    cfg = {
        axis: trial.suggest_categorical(axis, options)
        for axis, options in GROUP_AXES.items()
    }
    # 이 cfg로 전처리+LGBM 5-fold (HP는 default 고정, seed=42 고정 → 축 효과·상호작용만 학습)
    result = axes.run_one(
        cfg, seed=42, n_jobs=N_JOBS, n_estimators=N_ESTIMATORS,
    )
    # objective는 oof_rmse (train 5-fold CV) — strategy_common: best 탐색은 train OOF 기준.
    # val/test를 직접 최적화하면 cherry-picking이라 user_attr에 참고용으로만 기록.
    trial.set_user_attr('val_rmse',  result['val_rmse'])
    trial.set_user_attr('test_rmse', result['test_rmse'])
    trial.set_user_attr('elapsed_sec', result['elapsed_sec'])
    trial.set_user_attr('effective_target_transform', result['effective_target_transform'])  # tweedie loss면 'none' override 추적
    return result['oof_rmse']

# trial은 직렬(n_jobs=1)로 — 모델 내부 N_JOBS와 곱해져 코어가 과할당되는 걸 막음
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)


  0%|          | 0/300 [00:00<?, ?it/s]

Rerun: using best_pp_params_resolved (from trial.user_attrs) — conditional skip 키도 보존됨
Rerun preprocessing: cleaning=12 args, outlier method=iqr_clip, binarize_apply=True, iso_enabled=True, lds_enabled=True, ge_use_encoder=False, agg_funcs=['mean', 'std']
클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 950개
    컬럼: 977 → 950 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 347개, 잔여: 603개
    컬럼: 950 → 603 (347개 제거)
    DataFrame: (104748, 607)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[결측 imputation] method=median
  imputation 후 잔여 결측: 0

[고상관 제거] threshold=0.98, keep_by=std (std)
  제거: 0개, 잔여: 603개
    [고상관 제거 2차 / imputation 후] threshold=0.98
    컬럼: 603 → 603 (0개 제거)
    DataFrame: (104748, 616)

클리닝 완료: 1087 → 603 features

## 4. 산출물 저장

- `trials.csv` — study.trials_dataframe()
- `param_importance.csv` — fANOVA 기반 축 중요도 (OAT tornado와 비교용)

optuna.db는 storage로 자동 저장됨.

In [4]:
import pandas as pd
from optuna.importance import get_param_importances, FanovaImportanceEvaluator

# 전체 trial 표를 csv로
trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(OUT_DIR, 'trials.csv'), index=False)

# fANOVA 기반 축 중요도 — OAT tornado(축별 marginal)와 비교용
imp = get_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=42))
imp_df = pd.DataFrame([{'axis': k, 'importance': v} for k, v in imp.items()])
imp_df.to_csv(os.path.join(OUT_DIR, 'param_importance.csv'), index=False)

print(f'trials.csv         : {len(trials_df)} rows → {OUT_DIR}/trials.csv')
print(f'param_importance   : {len(imp_df)} axes  → {OUT_DIR}/param_importance.csv')

trials.csv         : 300 rows → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\baseline\group/trials.csv
param_importance   : 11 axes  → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\baseline\group/param_importance.csv
